# Targeted multimodal pilot — Phase A: mouse selection
Choose 5–10 seeds using visual evidence only. This notebook reuses the 25-image inventory. It never chooses target seeds or biological states automatically, and does not inspect audio.

Use the project `.venv` kernel. For click-and-drag inside VS Code/Jupyter, install `ipympl` from `requirements.txt` if needed and restart the kernel. If the plot remains static, use the OpenCV draw-window button (local graphical desktop required).


In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import pandas
import matplotlib
import cv2
import ipympl

print("Everything works")

In [ ]:
from pathlib import Path
import sys
from IPython import get_ipython
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Open this notebook inside the existing seed-germination project.')
sys.path.insert(0, str(PROJECT_ROOT))
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception as exc:
    print(f'Widget backend unavailable ({exc}). Use OpenCV draw-window fallback.')
    get_ipython().run_line_magic('matplotlib', 'inline')
from src.targeted_visual import load_pilot_inventory
from src.target_seed_interactive import SeedROIEditor
from scripts.prepare_target_seed_pilot import main
frames = load_pilot_inventory(PROJECT_ROOT / 'results/tables/dataset_inventory.csv')
print(f'{len(frames)} pilot frames: {frames.timestamp.iloc[0]} → {frames.timestamp.iloc[-1]}')
print(f'Kernel Python: {sys.executable}')
main(root=PROJECT_ROOT)


## 1. Select seed rectangles with the mouse
Run the next cell. The middle pilot frame is the default reference; change it with the **Reference** dropdown. The displayed title shows filename, exact timestamp and image dimensions.

1. Turn off toolbar pan/zoom, then drag around one seed on the live image. If the image stays static, click **OpenCV draw window**, drag there, then press Enter or Space (C cancels).
2. Choose an unused `seed_01`–`seed_10` ID, adjust **Padding px** (default 20), then click **Add rectangle**. Red is the manually drawn rectangle; cyan is the effective padded crop. Include emerging tissue but avoid a crop so broad that neighbors dominate.
3. Repeat for 5–10 visually clear seeds. To fix one, choose its ID under **Saved in UI**, draw a new box if needed, then click **Update selected**. **Delete selected** removes one; **Undo** reverses the last edit. **Reset all** requires its own confirmation checkbox. These changes stay in memory until saved.
4. Click **Save selections**. If the CSV already contains selections, tick the explicit replacement checkbox; a backup is made. Then click **Generate PNGs + GIFs**. Rerun this cell to reload existing selections. No Python code or pixel-coordinate entry is needed.

Check the camera-stability message above. Meaningful/weak estimates need human inspection across early, middle and late crops. Fixed coordinates are never translated automatically.


In [ ]:
editor = SeedROIEditor(PROJECT_ROOT, frames)


## 2. Inspect timelines and seed identity
After Save and Generate, inspect `selected_seed_overview.png`, each `seed_timelines/seed_XX_timeline.png` and `.gif`, and `selected_seed_progression_overview.png`. GIF playback is **constant 700 ms per frame, not real-time proportional**; there is no interpolation. A fixed camera does not imply fixed biological seed position. Verify the same physical seed in each frame before entering states.


In [ ]:
print(PROJECT_ROOT / 'results/figures/targeted_multimodal/selected_seed_overview.png')
print(PROJECT_ROOT / 'results/figures/targeted_multimodal/seed_timelines')
print(PROJECT_ROOT / 'results/tables/targeted_multimodal/target_seed_visual_review.csv')


## 3. Optional manual per-frame recentering
Run the next cell **after** selecting and generating reference ROIs. Choose a seed and one pilot frame. The plot zooms near its current crop. If that biological seed moved, drag over its new position, then click **Apply recentering**. This moves the crop center but preserves the reference crop's width and height, including at image edges. Only this frame changes; other frames retain their reference ROI unless you explicitly correct them. Use **Restore reference** for an incorrect correction, or **Undo**. The OpenCV draw-window fallback also works here (full-frame view).

For each inspected seed/frame, click **Identity confirmed**, **Identity uncertain**, or **Not same seed**; an untouched row remains unreviewed. Do not confirm merely because something occupies the same coordinates. Tick replacement confirmation, **Save corrections**, then **Regenerate timelines**. The table `target_seed_frame_rois.csv` retains all 25 effective ROIs per seed, their source (`reference` or `manual_correction`), and your identity review. Existing corrections are loaded, not erased. If a change conflicts with already-entered visual states, regeneration stops to protect those human observations.


In [ ]:
from src.target_seed_interactive import FrameROICorrectionEditor
correction_editor = FrameROICorrectionEditor(PROJECT_ROOT, frames)


## 4. Human visual states and event intervals
After checking identity and any needed recentering, open `target_seed_visual_review.csv`. It has one row per seed and frame, exact timestamp, the effective ROI, `roi_source`, and your identity review. Fill only `manual_state`, `manual_confidence`, and `notes` manually. States: `intact`, `possible_crack`, `clear_crack`, `radicle_visible`, `advanced_germination`, `occluded`, `uncertain`. Set seed priority (`high`, `medium`, `exclude`) visually in the ROI editor.

Only after that review, enter clear `crack` and/or `radicle_emergence` decisions in `target_seed_event_review.csv`: seed ID, event type, adjacent last-negative and first-positive pilot indices, human priority, confidence and notes. Then run `main(root=PROJECT_ROOT)` again. An event needs manually reviewed states **and confirmed identity in both boundary frames**. Ambiguous transitions stay unresolved. The interval is `(last_negative_timestamp, first_positive_timestamp]`; first positive is **not** the exact biological event time. Do not inspect audio in Phase A.
